<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.3-model-selection-and-routing/notebooks/exercises-11.3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.3 — Model Selection & Routing
Netsetos GenAI Course v2.0 | Module 11: Cost & Performance

Choose default tier, RouteLLM, cascading vs cost-aware vs semantic, routing eval in CI, FT vs route, Sarvam for India.


In [ ]:
print('Module 11: Cost & Performance')
print('Lesson 11.3: Model Selection & Routing')


## Ex 1: 2026 Model Tier Map


In [ ]:
print('When to pick which tier:')
for tier, models, ideal_for in [
    ('Nano',  'Gemini Flash, GPT-5 Nano, Claude Haiku',     'classification, FAQ, intent'),
    ('Mid',   'GPT-4o, Sonnet 4.6, Gemini 2.5 Pro',         'production reasoning, RAG'),
    ('Front', 'Opus 4.7, GPT-5.3 Pro',                      'complex planning, hard reasoning'),
    ('FT-OS', 'Llama-3.1-8B FT, Sarvam-M FT',                'narrow stable task, 100k+ req/mo'),
    ('Specl', 'Sarvam (Indic), code-llama, math-llama',     'specialist domains'),
]:
    print(f'  {tier:6s}: {models:42s} | ideal: {ideal_for}')


## Ex 2: Routing Strategies


In [ ]:
print('Routing strategies compared:')
for strat, how, latency_cost in [
    ('Cost-aware classifier','Classify upfront, route once',   '+0 RTT'),
    ('Cascading',           'Try cheap, escalate on low-conf', '+1 RTT on cascade'),
    ('Semantic',            'Embed prompt, route to specialist','+embedding lookup'),
    ('LLM-as-judge route',  'Strong model picks who answers',   '+1 RTT (paid)'),
    ('RouteLLM (Ong et al)','Trained classifier, BERT-based',   '+1 ms'),
]:
    print(f'  {strat:24s}: {how:36s} | {latency_cost}')


## Ex 3: RouteLLM Cost Math


In [ ]:
print('RouteLLM 70/22/8 split (Flash/Haiku/Sonnet):')
PRICES = {"flash": 0.40, "haiku": 4.00, "sonnet": 15.00}  # output $/MTok
split = {"flash": 0.70, "haiku": 0.22, "sonnet": 0.08}
weighted = sum(PRICES[m]*split[m] for m in PRICES)
baseline = PRICES['sonnet']
savings = (1 - weighted/baseline) * 100
print(f'  weighted output rate: ${weighted:.2f}/M')
print(f'  baseline (all sonnet): ${baseline:.2f}/M')
print(f'  savings: {savings:.1f}%')
print()
print('Quality drop typically -1 to -3pp on mixed traffic. Eval-gate the split.')


## Ex 4: Cascading with Confidence Threshold


In [ ]:
print('Cascading hit rate vs confidence threshold (illustrative):')
for tau, cheap_pct, escalate_pct, savings in [
    (0.5, 0.50, 0.50, '40%'),
    (0.7, 0.70, 0.30, '55%'),
    (0.85, 0.85, 0.15, '68%'),
    (0.95, 0.95, 0.05, '76%'),
]:
    print(f'  threshold={tau:.2f} | cheap={cheap_pct:.0%} escalate={escalate_pct:.0%} | est savings: {savings}')
print()
print('Higher threshold = more cheap calls + more cascade misses. Tune on labeled set.')


## Ex 5: Provider Strengths Matrix


In [ ]:
print('Provider strengths (bench on YOUR data, but starting points):')
for prov, strength, weakness in [
    ('Anthropic', 'Long context stable, nuanced refusals, code', 'Slower TTFT than OpenAI'),
    ('OpenAI',    'Structured output, function calling, voice', 'Cost on long contexts'),
    ('Gemini',    'Cheapest scale, multimodal, GCP integration','Newer ecosystem'),
    ('Mistral',   'Open weights, EU residency',                  'Behind on quality at top'),
    ('Sarvam',    'Indic languages +86% Indian math',            'English-only weaker'),
]:
    print(f'  {prov:10s}: + {strength:48s} | - {weakness}')


## Ex 6: Routing-Eval Snapshot


In [ ]:
print('Held-out routing-eval (200 prompts, ground-truth tier):')
for prompt_class, count, correct_tier_pct in [
    ('FAQ / classification',  80, 96),
    ('Mid reasoning',         70, 88),
    ('Complex planning',      30, 82),
    ('Indic language',        20, 91),
]:
    print(f'  {prompt_class:24s} n={count:>3} | router accuracy: {correct_tier_pct}%')
print()
print('Block PR when overall accuracy drops > 5pp.')


## Ex 7: Fine-Tune vs Route Cross-Over


In [ ]:
print('When does fine-tuned 8B beat routed Sonnet?')
for monthly_calls, route_cost, ft_cost, winner in [
    ( 30_000,  450,  900, 'route'),
    (100_000, 1500, 1100, 'FT'),
    (500_000, 7500, 1500, 'FT'),
    ( 10_000,  150, 1000, 'route'),
]:
    print(f'  {monthly_calls:>7,}/mo | route ${route_cost:>5,} vs FT ${ft_cost:>5,} | winner: {winner}')
print()
print('Cross-over ~80-150k req/mo for narrow tasks; assumes 1k+ labeled examples + stable scope.')


---
## Recap
Default to cheapest tier passing eval. Pin versions. Cache routing decisions.
Cost-aware (fast) vs cascading (best quality) vs semantic (specialist domains).
RouteLLM 70/22/8 = ~75% savings, -1 to -3pp quality. Eval-gate routing accuracy.
FT 8B above ~100k req/mo for narrow stable tasks. Sarvam for Indic content.
